# Reproduce a Quanthecy signal

This notebook uses the checked-in **synthetic** 15-minute fixture by default. It makes no network or model calls. Run from a Python 3.12 environment after `uv sync --frozen`. To inspect a real signal, download its CSV/Parquet evidence from the market detail page and set `input_path` below. The `envelope_json` column preserves the complete normalized input. These are observed REST midpoints, not trades.

In [ ]:
import json
import sys
from pathlib import Path

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists():
    if root == root.parent:
        raise RuntimeError("Run inside the Quanthecy checkout")
    root = root.parent
sys.path.insert(0, str(root / "python"))

In [ ]:
import polars as pl
from quanthecy_analytics.contracts.validation import validate_observation
from quanthecy_analytics.exports import export_observations
from quanthecy_analytics.signals import PARAMETERS, VERSION, analyze

In [ ]:
input_path = root / "tests/fixtures/research-window.json"
if input_path.suffix == ".json":
    observations = json.loads(input_path.read_text())
else:
    frame = (
        pl.read_parquet(input_path) if input_path.suffix == ".parquet" else pl.read_csv(input_path)
    )
    observations = [json.loads(value) for value in frame["envelope_json"]]
for observation in observations:
    validate_observation(json.dumps(observation))
metrics, signals = analyze(observations)
print(VERSION, PARAMETERS)
print(json.dumps(metrics, indent=2))
print(json.dumps(signals, indent=2))

In [ ]:
import io

for file_format in ("csv", "parquet"):
    exported = export_observations(observations, file_format)
    reader = pl.read_csv if file_format == "csv" else pl.read_parquet
    restored = [json.loads(value) for value in reader(io.BytesIO(exported))["envelope_json"]]
    assert analyze(restored) == (metrics, signals)

if input_path.name == "research-window.json":
    assert {signal["signal_type"] for signal in signals} == {
        "PROBABILITY_SPIKE",
        "SPREAD_WIDENING",
        "VOLUME_SPIKE",
    }
print("Both export formats reproduce identical metrics and signal IDs.")

## Reading the result

Probability and spread changes are fractions: `0.05` means **5 percentage points**. Volume rates use the exchange-reported cumulative volume and observation-time intervals; units remain USD for Polymarket and contracts for Kalshi. The z-score compares the last interval against preceding intervals. Missing quotes, counter resets, gaps and rule changes suppress affected calculations. A constant volume baseline has an undefined z-score.

Each signal contains its calculation version, parameters, window, and exact observation IDs. The synthetic fixture deliberately triggers all three signal families; it is not market evidence. Live data may correctly produce no signals.